<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 3 (Miércoles) — Clasificación 2 — Redes neuronales artificiales**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🔢 Red neuronal artificial: reconocer dígitos escritos a mano (MNIST)

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Reconocer dígitos manuscritos (OCR): convertir la imagen de un número en el número. |
| 🔍 Comprensión de los datos | Cargamos MNIST con `keras` (60 000 imágenes de entrenamiento y 10 000 de prueba, de 28×28 píxeles). |
| 🧹 Preparación de los datos | Escalamos los píxeles a [0, 1] y aplanamos cada imagen a 784 variables. |
| 🤖 Modelado | Entrenamos un perceptrón multicapa (`MLPClassifier`, capas 50-25-15-7-3, activación `tanh`, optimizador Adam). |
| 📊 Evaluación | Exactitud en prueba, matriz de confusión e informe por dígito. |
| 🚀 Despliegue y conclusiones | Visualizamos predicciones sobre imágenes de prueba al azar. |

> 🎯 **Analogía:** un cartero que aprende a leer números en los sobres: tras ver 60 000 ejemplos, cada neurona se especializa en un trazo (un arco, una barra) y la red combina esos trazos para decidir el dígito.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 👉 **3** | Miércoles | **Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales** |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

---
## 🎯 Fase 1 · Comprensión del negocio

Reconocer dígitos manuscritos (OCR): convertir la imagen de un número en el número.

📚 Lectura recomendada: [5]

## Análisis del problema

El **reconocimiento óptico de caracteres (OCR)** extrae el texto de una imagen (una foto o un documento escaneado) y lo convierte en cadenas de caracteres que se pueden editar. Aquí abordamos su versión más pequeña: **reconocer dígitos escritos a mano**, una tarea clásica para estudiar redes neuronales [5].

# Clasificador de digitos

Las redes neuronales artificiales (también conocidas como sistemas conexionistas) son un modelo computacional vagamente inspirado en el comportamiento observado en su homólogo biológico1.
Consiste en un conjunto de unidades, llamadas neuronas artificiales, conectadas entre sí para transmitirse señales. La información de entrada atraviesa la red neuronal (donde se somete a diversas operaciones) produciendo unos valores de salida.

### 🧪 Teoría rápida: ¿cómo aprende una red neuronal?

Una **neurona artificial** calcula una suma ponderada de sus entradas, le suma un sesgo y aplica una **función de activación** [3]. Muchas neuronas en capas forman un **perceptrón multicapa**: capa de entrada (aquí, 784 píxeles), capas ocultas y capa de salida (10 dígitos).

![Neurona artificial](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch12/images/12_01.png)
*Figura: neurona artificial. [11]*

![Perceptrón multicapa](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch12/images/12_02.png)
*Figura: perceptrón multicapa. [11]*

El entrenamiento repite dos pasos: **propagación hacia adelante** (calcular la predicción y la pérdida) y **retropropagación** (calcular cómo cambiar cada peso para reducir la pérdida) [4]. Los pesos se actualizan con un optimizador como Adam [6].

![Retropropagación](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch12/images/12_12.png)
*Figura: el error de salida se propaga hacia atrás. [11]*

**MNIST** es el conjunto clásico de dígitos escritos a mano [5].

![Ejemplos de MNIST](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch12/images/12_05.png)
*Figura: ejemplos de dígitos MNIST. [11]*

### Librerías

In [ ]:
import warnings
warnings.filterwarnings('ignore')
#imports
#numpy,pandas,scipy, math, matplotlib
import numpy as np
import pandas as pd
import scipy
from math import sqrt
import matplotlib.pyplot as plt
import seaborn as sns

from keras.datasets import mnist


#model metrics
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn import metrics

#cross validation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score

from numpy import random as rnd

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import plotly.graph_objects as go

---
## 🔍 Fase 2 · Comprensión de los datos

Cargamos MNIST con `keras` (60 000 imágenes de entrenamiento y 10 000 de prueba, de 28×28 píxeles).

📚 Lectura recomendada: [5]

## Entendimiento de los Datos

El total es de 70000 registros con imagenes de numeros

In [ ]:
#Cargar los datos
# load dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

#X_train = X_train[:60000,:,:]
#X_test = X_test[:10000,:,:]
#y_train = y_train[:60000]
#y_test  = y_test[:10000]



In [ ]:
# Numero de registros del dataset
print("Cantidad de registros:",len(y_train)+len(y_test))
# Tipos de datos del dataset

---
## 🔍 Fase 2 · Comprensión de los datos — explorar los dígitos

Cargamos MNIST con `keras` (60 000 imágenes de entrenamiento y 10 000 de prueba, de 28×28 píxeles).

## Exploración de los Datos

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(5, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'Etiqueta: {y_train[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
print("Entrenamiento dimensiones",X_train.shape)

In [ ]:
print("Pruebas dimensiones",X_test.shape)

---
## 🧹 Fase 3 · Preparación de los datos

Escalamos los píxeles a [0, 1] y aplanamos cada imagen a 784 variables.

📚 Lectura recomendada: [7]

In [ ]:
X_train=X_train/255.0
X_test=X_test/255.0

X= np.array(X_train).reshape(len(X_train),-1)

---
## 🤖 Fase 4 · Modelado

Entrenamos un perceptrón multicapa (`MLPClassifier`, capas 50-25-15-7-3, activación `tanh`, optimizador Adam).

📚 Lectura recomendada: [4] [6]

## Modelo de Machine Learning

Una vez seleccionadas las variables para incluir en el modelo de regresión, se procede a crearlo:

In [ ]:
#--------------
# ANN
#--------------
from sklearn.neural_network import MLPClassifier
#model = MLPClassifier(activation='tanh', max_iter=10000, hidden_layer_sizes=(28,14,7,3,1),random_state=1,
#                    learning_rate_init=0.001,verbose=10)
model = MLPClassifier(hidden_layer_sizes=(50,25,15,7,3), max_iter=10,
                    solver='adam', verbose=10, random_state=1,activation='tanh',
                    learning_rate_init=0.01)

In [ ]:
# Entrenamos nuestro modelo
model.fit(X, np.array(y_train))

In [ ]:
# Obtener el historial de entrenamiento del modelo
history = model.loss_curve_
# Crear figura con el gráfico de loss
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(history))),
                         y=history,
                         mode='lines',
                         name='Loss'))

fig.update_layout(title='Gráfico de Loss',
                  xaxis_title='Iteraciones',
                  yaxis_title='Loss')

# Mostrar figura
fig.show()

In [ ]:
# Arquitectura de la red entrenada: forma de la matriz de pesos entre capas consecutivas
for i, w in enumerate(model.coefs_):
    print(f"Capa {i} → capa {i+1}: matriz de pesos {w.shape[0]} × {w.shape[1]}")
print("Total de parámetros (pesos):", sum(w.size for w in model.coefs_))

In [ ]:
#fig, axes = plt.subplots(10, 4)
fig, axes = plt.subplots(5, 10)
# use global min / max to ensure all weights are shown on the same scale
vmin, vmax = model.coefs_[0].min(), model.coefs_[0].max()
for coef, ax in zip(model.coefs_[0].T, axes.ravel()):
    ax.matshow(coef.reshape(28, 28), cmap=plt.cm.gray, vmin=.5 * vmin,
               vmax=.5 * vmax)
    ax.set_xticks(())
    ax.set_yticks(())

plt.show()

In [ ]:
# Hacemos las predicciones que en definitiva una línea (en este caso, al ser 2D)
XTEST= np.array(X_test).reshape(len(X_test),-1)
y_pred1 = model.predict( XTEST )

In [ ]:
y_pred1

In [ ]:
predict_values = model.predict_proba(XTEST)
print("Probabilidad de cada dígito (0 a 9) para la primera imagen de prueba:")
print(np.round(predict_values[0], 3), "→ predicción:", predict_values[0].argmax(), "| real:", y_test[0])

---
## 📊 Fase 5 · Evaluación

Exactitud en prueba, matriz de confusión e informe por dígito.

📚 Lectura recomendada: [10]

Evaluación

In [ ]:
print("Exactitud en prueba: %f" % model.score(XTEST, y_test))

In [ ]:
matrix=confusion_matrix(np.array(y_test), np.array(y_pred1))
matrix

In [ ]:
figure = plt.figure(figsize=(8, 8))
sns.heatmap(matrix, annot=True, fmt='g', cmap=plt.cm.Blues)
plt.tight_layout()
plt.ylabel('Dígito real')
plt.xlabel('Dígito predicho')
plt.show()

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(np.array(y_test), np.array(y_pred1)))

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=matrix,display_labels=model.classes_)
disp.plot()

---
## 🚀 Fase 6 · Despliegue y conclusiones — predecir dígitos de prueba al azar

Visualizamos predicciones sobre imágenes de prueba al azar.

📚 Lectura recomendada: [5]

In [ ]:
# Select n_sel random images from the test dataset -- we have 10,000 test images in total (see above)
n_total = 10000
n_sel = 10
test_image_ids = rnd.choice(range(0,n_total - 1), n_sel, replace=False)

# for each of these image ids, get the test image
test_images_sel = [X_test[i] for i in test_image_ids]
test_images_sel= np.array(test_images_sel).reshape(len(test_images_sel),-1)
# for each of these image ids, get the result from the classifier
model.predict(test_images_sel)

In [ ]:
# Now show these randomly selected test images and our neural net classifier's prediction based on its training
# Adapted from Sonya Sawtelle
# http://sdsawtelle.github.io/blog/output/week4-andrew-ng-machine-learning-with-python.html
fig, axs = plt.subplots(2, 5, sharex=False, sharey=False, figsize=(10,10))
axs = axs.flatten()  # The returned axs is actually a matrix holding the handles to all the subplot axes objects
graymap = plt.get_cmap("gray")

for i, indx in enumerate(test_image_ids):
  im_mat = np.reshape(test_images_sel[i], (28, 28))
  labl =model.predict(test_images_sel[i].reshape(1, -1))

  # Plot the image along with the label it is assigned by the fitted model.
  axs[i].imshow(im_mat, cmap=graymap, interpolation="None")
  axs[i].annotate(str(labl[0]), xy=(0.05, 0.85), xycoords="axes fraction", color="red", fontsize=18)
  axs[i].xaxis.set_visible(False)  # Hide the axes labels for clarity
  axs[i].yaxis.set_visible(False)

---
## 🧪 Refuerzo: ¿cuántas neuronas y por qué importa escalar los píxeles?

El tamaño de la red (número de capas y neuronas) y la **escala de las entradas** influyen mucho en el aprendizaje. Con píxeles entre 0 y 255 la suma ponderada de la primera capa es enorme, la activación `tanh` se **satura** (derivada ≈ 0) y el descenso de gradiente casi no avanza; por eso se divide entre 255 [7]. Comparamos varias arquitecturas con el optimizador Adam [6] usando 10 épocas, y la misma red sin escalar.

In [ ]:
import time
from sklearn.neural_network import MLPClassifier

filas = []
for ocultas in [(3,), (16,), (50, 25, 15, 7, 3), (128, 64)]:
    t0 = time.time()
    m = MLPClassifier(hidden_layer_sizes=ocultas, max_iter=10, solver='adam', random_state=1, activation='tanh', learning_rate_init=0.01).fit(X, y_train)
    filas.append({'arquitectura': str(ocultas), 'parámetros': sum(w.size for w in m.coefs_), 'exactitud (prueba)': round(m.score(XTEST, y_test), 4), 'segundos': round(time.time() - t0, 1)})

# La misma red con los píxeles SIN escalar (0–255)
m_sin = MLPClassifier(hidden_layer_sizes=(50, 25, 15, 7, 3), max_iter=10, solver='adam', random_state=1, activation='tanh', learning_rate_init=0.01).fit(X * 255, y_train)
filas.append({'arquitectura': '(50, 25, 15, 7, 3) sin escalar', 'parámetros': sum(w.size for w in m_sin.coefs_), 'exactitud (prueba)': round(m_sin.score(XTEST * 255, y_test), 4), 'segundos': np.nan})
pd.DataFrame(filas)

### ✍️ Ejercicios
1. ¿Qué pasa con la exactitud cuando la capa oculta tiene solo 3 neuronas (≈ 74 %)? ¿Por qué comprimir 784 píxeles en 3 números pierde tanta información?
2. Cambie `activation` a `'relu'` y compare con `'tanh'` en la red sin escalar.
3. Identifique en la matriz de confusión los pares de dígitos más confundidos (p. ej. 4↔9, 3↔5) y explique por qué.

---
## ✅ Conclusiones y siguientes pasos

Una red pequeña (50-25-15-7-3) ya alcanza una exactitud alta en MNIST con solo 10 épocas, aunque confunde dígitos de trazo parecido (4↔9, 3↔5, 7↔2). Una red más ancha mejora el resultado y, para imágenes, las **redes convolucionales** superan a los perceptrones multicapa [5]. Escalar las entradas y vigilar la curva de pérdida son pasos imprescindibles [7].

---
## 📚 Referencias

Fuentes adicionales para profundizar: [8], [9].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Rosenblatt, F. (1958). The perceptron: A probabilistic model for information storage and organization in the brain. *Psychological Review, 65*(6), 386–408. https://doi.org/10.1037/h0042519

**[4]** Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). Learning representations by back-propagating errors. *Nature, 323*(6088), 533–536. https://doi.org/10.1038/323533a0

**[5]** LeCun, Y., Bottou, L., Bengio, Y., & Haffner, P. (1998). Gradient-based learning applied to document recognition. *Proceedings of the IEEE, 86*(11), 2278–2324. https://doi.org/10.1109/5.726791

**[6]** Kingma, D. P., & Ba, J. (2015). Adam: A method for stochastic optimization. *3rd International Conference on Learning Representations (ICLR)*. https://arxiv.org/abs/1412.6980

**[7]** Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press. https://www.deeplearningbook.org

**[8]** Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3.ª ed.). O'Reilly Media.

**[9]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

**[10]** Powers, D. M. W. (2011). Evaluation: From precision, recall and F-measure to ROC, informedness, markedness and correlation. *Journal of Machine Learning Technologies, 2*(1), 37–63.

**[11]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)